# Preparation des donnees — Modele CLV Nouvelles Acquisitions

## Objectif

Lire, nettoyer et consolider les fichiers bruts (`RAW_DATA/`) pour produire des fichiers propres prets pour l'EDA et la verification des hypotheses.

**Les noms de colonnes sont standardises** a la lecture : accents retires, espaces → underscores, snake_case.
Ex. `Revenus EUR avec Liquidity` → `revenus_eur_avec_liquidity`, `CP Dépôts à terme` → `cp_depots_a_terme`.

## Structure des donnees brutes

```
RAW_DATA/
├── STOCK/         STOCK_EER_18..25.csv   → declaratif client (segment, profession, age, etc.)
├── FLUX/          FLUX_18..25.csv        → avoirs, volumes, NBI total mensuel
└── PRODUITS/      PRODUITS_18..25/
                   ├── BANCA_99.csv       → bancassurance (assurance placement, Optipension)
                   ├── CARTES_99.csv      → cartes (commissions, transactions)
                   ├── CREDIT_99.csv      → credits (prets perso, etudes, credits a vue)
                   ├── INVEST_99.csv      → investissements (depots a terme, titres)
                   ├── PRET_IMMO_99.csv   → prets immobiliers
                   └── SAVINGS_99.csv     → comptes courants / epargne
```

Chaque fichier = snapshots mensuels par client, cles par `(identifiant_tiers, periode)`.
`periode` = indice mois bancaire (Janvier 1994 = 1).

## Mapping vers les 5 familles du modele

| Fichier(s) | Famille | Encours | NBI |
|------------|---------|---------|-----|
| SAVINGS | 1 — Epargne bilan | avoirs_eur | revenus_eur_avec_liquidity |
| INVEST + BANCA | 2 — Investissements | cp_* / capitaux_productifs_eur | revenus_eur_avec_liquidity |
| PRET_IMMO | 3 — Credit immobilier | credits_utilises_eur | revenus_eur_avec_liquidity |
| CREDIT | 4 — Autres credits | credits_utilises_eur | revenus_eur_avec_liquidity |
| CARTES | 5 — Autre PNB | N/A (NBI direct) | revenus_eur_avec_liquidity |

FLUX fournit le **NBI total** par client par mois → utilise pour validation.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Configuration ────────────────────────────────────────────────────────────
RAW_DATA = Path("../../RAW_DATA")
OUTPUT_DIR = Path("../clean_data")
OUTPUT_DIR.mkdir(exist_ok=True)

YEARS = list(range(18, 26))  # 18=2018 .. 25=2025
EPOCH_YEAR = 1994

# Codes NACE agriculture a exclure — A REMPLIR PAR L'UTILISATEUR
NACE_AGRICULTURE_CODES = {
    # Exemples : "01.1", "01.2", "02.1", "03.1", etc.
    # Ajouter ici les codes NACE qui identifient les clients agricoles
}

PRODUCT_FILES = ["SAVINGS_99", "INVEST_99", "BANCA_99", "CARTES_99", "CREDIT_99", "PRET_IMMO_99"]

# CSV parameters (adjust if needed)
CSV_PARAMS = dict(delimiter=";", encoding="latin-1")


# ── Column name standardization ─────────────────────────────────────────────
def standardize_col(name: str) -> str:
    """'CP Dépôts à terme' -> 'cp_depots_a_terme'"""
    s = unicodedata.normalize("NFD", name)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[/()\.\-,']", " ", s)
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def standardize_columns(df):
    df.columns = [standardize_col(c) for c in df.columns]
    return df


# ── Helpers ──────────────────────────────────────────────────────────────────
def idx_to_yearmonth(idx):
    year = EPOCH_YEAR + (idx - 1) // 12
    month = (idx - 1) % 12 + 1
    return year, month

def yearmonth_to_idx(year, month):
    return (year - EPOCH_YEAR) * 12 + month

assert yearmonth_to_idx(1994, 1) == 1
assert yearmonth_to_idx(2025, 12) == 384
assert idx_to_yearmonth(384) == (2025, 12)


def parse_eur(series):
    """Convert European-formatted string amounts to float.
    Handles: '350,090.43' (comma thousands, dot decimal) and '350 090,43' (space thousands, comma decimal)."""
    if series.dtype in (np.float64, np.int64, float, int):
        return series.astype(float)
    s = series.astype(str)
    sample = s.dropna().head(100)
    has_comma_and_dot = sample.str.contains(r",.*\.", regex=True).any()
    if has_comma_and_dot:
        s = s.str.replace(",", "", regex=False)
    else:
        s = s.str.replace(" ", "", regex=False).str.replace("\xa0", "", regex=False)
        s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def standardize_id(series):
    """Convert Identifiant Tiers to nullable Int64, dropping NaN."""
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def load_csv(path, **kwargs):
    """Try multiple encodings to read a CSV, then standardize column names."""
    params = {**CSV_PARAMS, **kwargs}
    for enc in [params.get("encoding", "latin-1"), "utf-8", "cp1252"]:
        try:
            df = pd.read_csv(path, **{**params, "encoding": enc})
            return standardize_columns(df)
        except (UnicodeDecodeError, FileNotFoundError):
            continue
    raise ValueError(f"Cannot read: {path}")


print(f"RAW_DATA: {RAW_DATA.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"Years: 20{YEARS[0]} - 20{YEARS[-1]}")
print(f"NACE agriculture codes: {'PLACEHOLDER — a remplir' if not NACE_AGRICULTURE_CODES else len(NACE_AGRICULTURE_CODES)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Lecture et concatenation des fichiers STOCK
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("LECTURE STOCK")
print("=" * 70)

stock_frames = []
for yy in YEARS:
    path = RAW_DATA / "STOCK" / f"STOCK_EER_{yy}.csv"
    if not path.exists():
        print(f"  MANQUANT: {path.name}")
        continue
    df = load_csv(path)
    df["annee_acquisition"] = 2000 + yy
    stock_frames.append(df)
    print(f"  {path.name}: {len(df):,} lignes, {df['identifiant_tiers'].nunique():,.0f} clients uniques")

stock = pd.concat(stock_frames, ignore_index=True)

# Standardize types
stock["identifiant_tiers"] = standardize_id(stock["identifiant_tiers"])
stock["periode"] = pd.to_numeric(stock["periode"], errors="coerce").astype("Int64")

# Parse EUR columns
for col in ["revenu_total", "volume_entrant_cvl"]:
    if col in stock.columns:
        stock[col] = parse_eur(stock[col])

# Drop rows with no client ID
n_before = len(stock)
stock = stock.dropna(subset=["identifiant_tiers"])
print(f"\nSTOCK total: {len(stock):,} lignes ({n_before - len(stock)} sans identifiant_tiers)")
print(f"Clients uniques: {stock['identifiant_tiers'].nunique():,}")
print(f"Periode: {stock['periode'].min()} - {stock['periode'].max()}")
print(f"\nPar annee d'acquisition:")
print(stock.groupby("annee_acquisition")["identifiant_tiers"].nunique().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Algorithme d'exclusion des clients
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("EXCLUSION DES CLIENTS NON ELIGIBLES")
print("=" * 70)

n_total = len(stock)
reasons = {}

# Rule 1: Nature Tiers = 'M' (personne morale)
mask_moral = stock["nature_tiers_m_p"].str.strip().str.upper() == "M"
reasons["Personne morale (M)"] = mask_moral.sum()

# Rule 2: Profession liberale
mask_prof = stock["type_de_profession"].str.strip().str.lower() == "profession libérale"
reasons["Profession liberale"] = mask_prof.sum()

# Rule 3: Code NACE agriculture (placeholder — user fills NACE_AGRICULTURE_CODES)
if NACE_AGRICULTURE_CODES:
    mask_agri = stock["code_nace_general_vue_statistique"].isin(NACE_AGRICULTURE_CODES)
else:
    mask_agri = pd.Series(False, index=stock.index)
reasons["Agriculture (NACE)"] = mask_agri.sum()

# Rule 4: Non-resident AND non-frontalier
mask_nonres = (
    (stock["resident_o_n"].str.strip().str.upper() == "N") &
    (stock["frontalier_o_n"].str.strip().str.upper() == "N")
)
reasons["Non-resident non-frontalier"] = mask_nonres.sum()

# Combined exclusion mask
mask_exclude = mask_moral | mask_prof | mask_agri | mask_nonres

print(f"Lignes totales: {n_total:,}")
print(f"\nRaisons d'exclusion (peuvent se chevaucher):")
for reason, count in reasons.items():
    print(f"  {reason}: {count:,} ({count/n_total*100:.1f}%)")
print(f"\nTotal lignes exclues: {mask_exclude.sum():,} ({mask_exclude.sum()/n_total*100:.1f}%)")

# Apply exclusion
stock_eligible = stock[~mask_exclude].copy()

# Handle clients who switch status: keep only consecutive eligible months
def keep_consecutive_eligible(group):
    group = group.sort_values("periode")
    periods = group["periode"].values
    diffs = np.diff(periods)
    return group

stock_eligible = stock_eligible.groupby("identifiant_tiers", group_keys=False).apply(keep_consecutive_eligible)

# Build eligible set for filtering other files
eligible_set = set(zip(stock_eligible["identifiant_tiers"], stock_eligible["periode"]))
eligible_clients = set(stock_eligible["identifiant_tiers"].unique())

print(f"\nApres exclusion:")
print(f"  Lignes eligibles: {len(stock_eligible):,}")
print(f"  Clients eligibles: {len(eligible_clients):,}")
print(f"\nPar annee:")
print(stock_eligible.groupby("annee_acquisition")["identifiant_tiers"].nunique().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Lecture et concatenation des fichiers FLUX
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("LECTURE FLUX")
print("=" * 70)

flux_frames = []
for yy in YEARS:
    path = RAW_DATA / "FLUX" / f"FLUX_{yy}.csv"
    if not path.exists():
        print(f"  MANQUANT: {path.name}")
        continue
    df = load_csv(path)
    flux_frames.append(df)
    print(f"  {path.name}: {len(df):,} lignes")

flux = pd.concat(flux_frames, ignore_index=True)

# Standardize types
flux["identifiant_tiers"] = standardize_id(flux["identifiant_tiers"])
flux["periode"] = pd.to_numeric(flux["periode"], errors="coerce").astype("Int64")

# Parse EUR columns
eur_cols_flux = ["avoirs_eur", "avoirs_hors_bancassurance_cvl",
                 "volume_entrant_cvl", "volume_sortant_cvl", "revenus_eur_avec_liquidity"]
for col in eur_cols_flux:
    if col in flux.columns:
        flux[col] = parse_eur(flux[col])

# Drop rows with no client ID, filter to eligible clients
flux = flux.dropna(subset=["identifiant_tiers"])
n_before = len(flux)
flux = flux[flux["identifiant_tiers"].isin(eligible_clients)]

print(f"\nFLUX total: {n_before:,} lignes")
print(f"Apres filtre clients eligibles: {len(flux):,} lignes")
print(f"Clients: {flux['identifiant_tiers'].nunique():,}")
print(f"Periode: {flux['periode'].min()} - {flux['periode'].max()}")
print(f"\nNBI (revenus_eur_avec_liquidity) — stats:")
print(flux["revenus_eur_avec_liquidity"].describe().round(2).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Lecture et concatenation des fichiers PRODUITS (6 types x 8 annees)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("LECTURE PRODUITS")
print("=" * 70)

products = {}

# Columns to keep as strings (not parse as EUR) — standardized names
STRING_COLS = {
    "identifiant_tiers", "date_creation", "date_douverture", "date_decheance",
    "date_debut_relation", "produits", "ref_contrat", "type_doperation",
    "type_de_destination_du_credit", "destination_du_credit", "nature_tiers_m_p",
    "code_sexe_tiers", "code_etat_civil", "segment_ameliore", "segment_upgrade",
    "type_de_profession", "code_nace_general_vue_statistique",
    "resident_o_n", "frontalier_o_n",
    "centre_de_responsabilite_fusionne_principal",
}

for product_name in PRODUCT_FILES:
    frames = []
    for yy in YEARS:
        folder = RAW_DATA / "PRODUITS" / f"PRODUITS_{yy}"
        path = folder / f"{product_name}.csv"
        if not path.exists():
            continue
        df = load_csv(path)
        df["annee_acquisition"] = 2000 + yy
        frames.append(df)

    if not frames:
        print(f"  {product_name}: AUCUN FICHIER TROUVE")
        continue

    combined = pd.concat(frames, ignore_index=True)

    # Standardize ID and Periode
    combined["identifiant_tiers"] = standardize_id(combined["identifiant_tiers"])
    combined["periode"] = pd.to_numeric(combined["periode"], errors="coerce").astype("Int64")

    # Parse all object columns that look like amounts
    for col in combined.columns:
        if combined[col].dtype == object and col not in STRING_COLS:
            combined[col] = parse_eur(combined[col])

    # Filter to eligible clients
    combined = combined.dropna(subset=["identifiant_tiers"])
    combined = combined[combined["identifiant_tiers"].isin(eligible_clients)]

    products[product_name] = combined
    n_clients = combined["identifiant_tiers"].nunique()
    n_rows = len(combined)
    period_min = combined["periode"].min()
    period_max = combined["periode"].max()
    nbi_col = "revenus_eur_avec_liquidity"
    nbi_fill = combined[nbi_col].notna().sum() if nbi_col in combined.columns else 0
    print(f"  {product_name:<18} {n_rows:>9,} lignes | {n_clients:>7,} clients | "
          f"Periode {period_min}-{period_max} | NBI rempli: {nbi_fill:,}/{n_rows:,}")

print(f"\n{len(products)} fichiers produits charges.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Resume des types et completude
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("RESUME DES TYPES ET COMPLETUDE")
print("=" * 70)

all_dfs = {"STOCK": stock_eligible, "FLUX": flux, **products}

for name, df in all_dfs.items():
    print(f"\n{'─' * 50}")
    print(f"{name}: {len(df):,} lignes x {len(df.columns)} colonnes")
    print(f"{'─' * 50}")

    for col in df.columns:
        dtype = str(df[col].dtype)
        pct_null = df[col].isna().mean() * 100
        flag = ""
        if dtype == "object" and col not in STRING_COLS:
            flag = " ⚠ OBJECT (devrait etre numerique ?)"
        if pct_null > 50:
            flag += f" ⚠ {pct_null:.0f}% null"
        print(f"  {col:<50} {dtype:<10} null={pct_null:5.1f}%{flag}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Sanity checks
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("SANITY CHECKS")
print("=" * 70)

# ── 1. NBI cross-check: sum of per-product NBI vs FLUX NBI ──────────────────
print("\n1. NBI CROSS-CHECK (somme produits vs FLUX)")
print("-" * 50)

nbi_col = "revenus_eur_avec_liquidity"
product_nbi_frames = []
for pname, pdf in products.items():
    if nbi_col not in pdf.columns:
        continue
    sub = pdf[["identifiant_tiers", "periode", nbi_col]].dropna(subset=[nbi_col])
    sub = sub.rename(columns={nbi_col: f"nbi_{pname}"})
    sub = sub.groupby(["identifiant_tiers", "periode"], as_index=False).sum()
    product_nbi_frames.append(sub)

if product_nbi_frames and len(flux) > 0:
    nbi_merged = flux[["identifiant_tiers", "periode", nbi_col]].copy()
    nbi_merged = nbi_merged.rename(columns={nbi_col: "nbi_flux"})
    for f in product_nbi_frames:
        nbi_merged = nbi_merged.merge(f, on=["identifiant_tiers", "periode"], how="left")

    nbi_product_cols = [c for c in nbi_merged.columns if c.startswith("nbi_") and c != "nbi_flux"]
    nbi_merged["nbi_sum_products"] = nbi_merged[nbi_product_cols].sum(axis=1)
    nbi_merged["nbi_gap"] = nbi_merged["nbi_flux"] - nbi_merged["nbi_sum_products"]
    nbi_merged["nbi_gap_pct"] = (nbi_merged["nbi_gap"] / nbi_merged["nbi_flux"].abs().clip(lower=1)) * 100

    has_both = nbi_merged["nbi_flux"].notna() & (nbi_merged["nbi_sum_products"] != 0)
    subset = nbi_merged[has_both]

    print(f"  Lignes avec NBI flux et produits: {len(subset):,}")
    print(f"  Ecart moyen: {subset['nbi_gap'].mean():,.2f} EUR")
    print(f"  Ecart median: {subset['nbi_gap'].median():,.2f} EUR")
    print(f"  Ecart moyen (%): {subset['nbi_gap_pct'].mean():.1f}%")
    print(f"  Clients avec ecart > 10%: {(subset['nbi_gap_pct'].abs() > 10).sum():,} / {len(subset):,}")
else:
    print("  Pas assez de donnees pour le cross-check NBI")

# ── 2. Client coverage ───────────────────────────────────────────────────────
print("\n2. COUVERTURE CLIENTS")
print("-" * 50)

stock_clients = set(stock_eligible["identifiant_tiers"].unique())
flux_clients = set(flux["identifiant_tiers"].unique())
product_clients = set()
for pdf in products.values():
    product_clients.update(pdf["identifiant_tiers"].unique())

print(f"  Clients STOCK eligible: {len(stock_clients):,}")
print(f"  Clients FLUX: {len(flux_clients):,}")
print(f"  Clients PRODUITS (union): {len(product_clients):,}")
print(f"  Dans STOCK mais pas FLUX: {len(stock_clients - flux_clients):,}")
print(f"  Dans STOCK mais aucun produit: {len(stock_clients - product_clients):,}")
print(f"  Dans PRODUITS mais pas STOCK: {len(product_clients - stock_clients):,}")

# ── 3. Duplicates ────────────────────────────────────────────────────────────
print("\n3. DOUBLONS (client_id, periode) PAR FICHIER")
print("-" * 50)
for name, df in all_dfs.items():
    if "periode" not in df.columns:
        continue
    n_dup = df.duplicated(subset=["identifiant_tiers", "periode"]).sum()
    flag = " ⚠ DOUBLONS" if n_dup > 0 else ""
    print(f"  {name:<18}: {n_dup:,} doublons{flag}")

# ── 4. Period range ──────────────────────────────────────────────────────────
print("\n4. PLAGE DE PERIODES PAR FICHIER")
print("-" * 50)
for name, df in all_dfs.items():
    if "periode" not in df.columns:
        continue
    pmin, pmax = df["periode"].min(), df["periode"].max()
    ym_min = idx_to_yearmonth(int(pmin)) if pd.notna(pmin) else ("?", "?")
    ym_max = idx_to_yearmonth(int(pmax)) if pd.notna(pmax) else ("?", "?")
    print(f"  {name:<18}: {pmin} - {pmax}  ({ym_min[0]}-{ym_min[1]:02d} a {ym_max[0]}-{ym_max[1]:02d})")

# ── 5. Encours signs ─────────────────────────────────────────────────────────
print("\n5. SIGNES DES ENCOURS")
print("-" * 50)
encours_checks = {
    "SAVINGS_99": "avoirs_eur",
    "CREDIT_99": "credits_utilises_eur",
    "PRET_IMMO_99": "credits_utilises_eur",
    "INVEST_99": "cp_depots_a_terme",
    "BANCA_99": "capitaux_productifs_eur",
}
for pname, col in encours_checks.items():
    if pname in products and col in products[pname].columns:
        vals = products[pname][col]
        n_neg = (vals < 0).sum()
        print(f"  {pname:<18} [{col}]: {n_neg:,} valeurs negatives" +
              (" ⚠" if n_neg > 0 else " ✓"))

# ── 6. Zero-inflation preview ────────────────────────────────────────────────
print("\n6. ZERO-INFLATION (% clients avec encours = 0 sur TOUTES leurs periodes)")
print("-" * 50)
for pname, col in encours_checks.items():
    if pname in products and col in products[pname].columns:
        client_max = products[pname].groupby("identifiant_tiers")[col].max()
        n_all_zero = (client_max == 0).sum()
        n_total = len(client_max)
        print(f"  {pname:<18}: {n_all_zero:,} / {n_total:,} ({n_all_zero/max(n_total,1)*100:.1f}%) clients toujours a zero")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Sauvegarde des fichiers nettoyes
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("SAUVEGARDE")
print("=" * 70)

# Save STOCK
stock_out = OUTPUT_DIR / "clean_stock.parquet"
stock_eligible.to_parquet(stock_out, index=False)
print(f"  {stock_out.name}: {len(stock_eligible):,} lignes")

# Save FLUX
flux_out = OUTPUT_DIR / "clean_flux.parquet"
flux.to_parquet(flux_out, index=False)
print(f"  {flux_out.name}: {len(flux):,} lignes")

# Save each product file
for pname, pdf in products.items():
    fname = f"clean_{pname.lower().replace('_99', '')}.parquet"
    out_path = OUTPUT_DIR / fname
    pdf.to_parquet(out_path, index=False)
    print(f"  {fname}: {len(pdf):,} lignes")

print(f"\nTous les fichiers sauvegardes dans: {OUTPUT_DIR.resolve()}")
print(f"\nUsage:")
print(f"  stock = pd.read_parquet('{OUTPUT_DIR}/clean_stock.parquet')")
print(f"  flux = pd.read_parquet('{OUTPUT_DIR}/clean_flux.parquet')")
print(f"  savings = pd.read_parquet('{OUTPUT_DIR}/clean_savings.parquet')")
print(f"  # etc.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Dashboard de synthese
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("DASHBOARD DE SYNTHESE")
print("=" * 70)

# Clients par annee d'acquisition
print("\n1. CLIENTS ELIGIBLES PAR ANNEE D'ACQUISITION")
print("-" * 50)
clients_per_year = stock_eligible.groupby("annee_acquisition")["identifiant_tiers"].nunique()
for year, n in clients_per_year.items():
    print(f"  {year}: {n:,} clients")
print(f"  TOTAL: {clients_per_year.sum():,}")

# Observations par client
print("\n2. NOMBRE D'OBSERVATIONS MENSUELLES PAR CLIENT")
print("-" * 50)
obs_per_client = stock_eligible.groupby("identifiant_tiers")["periode"].count()
print(f"  Median: {obs_per_client.median():.0f} mois")
print(f"  Min: {obs_per_client.min()} mois")
print(f"  Max: {obs_per_client.max()} mois")
print(f"  Q25: {obs_per_client.quantile(0.25):.0f} | Q75: {obs_per_client.quantile(0.75):.0f}")

# Profondeur de donnees par cohorte
print("\n3. PROFONDEUR PAR COHORTE (mois de donnees disponibles)")
print("-" * 50)
depth = stock_eligible.groupby("annee_acquisition").agg(
    n_clients=("identifiant_tiers", "nunique"),
    periode_min=("periode", "min"),
    periode_max=("periode", "max"),
).reset_index()
depth["profondeur_mois"] = depth["periode_max"] - depth["periode_min"] + 1
for _, row in depth.iterrows():
    ym_max = idx_to_yearmonth(int(row["periode_max"]))
    print(f"  {int(row['annee_acquisition'])}: {int(row['profondeur_mois']):>3} mois max "
          f"(jusqu'a {ym_max[0]}-{ym_max[1]:02d}) | {int(row['n_clients']):,} clients")

# Stats par produit
print("\n4. STATS PAR PRODUIT")
print("-" * 50)
nbi_col = "revenus_eur_avec_liquidity"
for pname, pdf in products.items():
    n_clients = pdf["identifiant_tiers"].nunique()
    pct_of_stock = n_clients / len(eligible_clients) * 100
    nbi_mean = pdf[nbi_col].mean() if nbi_col in pdf.columns else float("nan")
    print(f"  {pname:<18}: {n_clients:>7,} clients ({pct_of_stock:5.1f}% du stock) | "
          f"NBI moyen: {nbi_mean:>10,.2f} EUR")

# Segmentation
print("\n5. SEGMENTS (snapshot le plus recent par client)")
print("-" * 50)
if "segment_ameliore" in stock_eligible.columns:
    latest = stock_eligible.sort_values("periode").groupby("identifiant_tiers").last()
    print(latest["segment_ameliore"].value_counts().to_string())

# Residents vs Frontaliers
print("\n6. RESIDENTS vs FRONTALIERS")
print("-" * 50)
if "frontalier_o_n" in stock_eligible.columns:
    latest = stock_eligible.sort_values("periode").groupby("identifiant_tiers").last()
    front = latest["frontalier_o_n"].str.strip().str.upper().value_counts()
    print(front.to_string())

print("\n" + "=" * 70)
print("PRET POUR L'EDA ET LA VERIFICATION DES HYPOTHESES")
print("=" * 70)